# Chargement du jeu de données nettoyé

Le jeu de données nettoyé est chargé depuis le fichier CSV.

Étant donné que le format CSV ne conserve pas les types de données, une étape de conversion est réalisée afin de restaurer les types appropriés (dates, entiers avec valeurs manquantes, etc.) avant de procéder à l'enrichissement des données.

Cette étape garantit la cohérence des traitements et évite les erreurs lors des calculs des variables dérivées.

In [32]:
import pandas as pd
# Chargement du dataset nettoyé
df = pd.read_csv("dataset_ocp_social_2026_nettoye_finale_version2.csv")
# ==========================================================
# Restauration des types de données
# ==========================================================

# Colonnes de type date
date_columns = [
    "date_demande",
    "date_debut_sejour",
    "date_fin_sejour",
    "periode_classement_debut",
    "periode_classement_fin",
    "date_limite_demande",
    "date_limite_paiement",
    "date_traitement",
    "date_paiement",
    "date_tirage_voucher"
]

for col in date_columns:
    df[col] = pd.to_datetime(df[col], errors="coerce")


# Colonnes de type entier nullable (Int64)
nullable_int_columns = [
    "reference_paiement",
    "capacite_logement",
    "nombre_pieces"
]

for col in nullable_int_columns:
    df[col] = df[col].astype("Int64")

In [34]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 47 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   numero_demande                  5000 non-null   object        
 1   site                            5000 non-null   object        
 2   matricule                       5000 non-null   int64         
 3   nom_agent                       5000 non-null   object        
 4   prenom_agent                    5000 non-null   object        
 5   genre                           5000 non-null   object        
 6   situation_familiale             5000 non-null   object        
 7   categorie_professionnelle       5000 non-null   object        
 8   anciennete_annees               5000 non-null   int64         
 9   annees_depuis_dernier_sejour    5000 non-null   int64         
 10  nombre_points                   5000 non-null   int64         
 11  type

## Enrichissement 1 : Création de la colonne `delai_traitement_jours`

### Objectif

Cette étape consiste à créer une nouvelle variable métier représentant le **délai de traitement d'une demande de séjour**.

Le délai de traitement correspond au nombre de jours écoulés entre :

- **`date_demande`** : date de soumission de la demande.
- **`date_traitement`** : date de traitement de la demande par l'administration.

### Formule

$$\text{délai\_traitement\_jours} = \text{date\_traitement} - \text{date\_demande}$$

### Intérêt métier

Cette variable constitue un indicateur de performance du processus de traitement des demandes.

Elle sera utilisée par la suite pour :

- analyser le temps moyen de traitement des demandes ;
- comparer les délais selon  les périodes ou les saisons ;
- construire des indicateurs (KPI) dans Power BI ;
- servir de variable explicative dans les modèles de Machine Learning.

In [35]:
# ==========================================================
# Enrichissement 1 : Création de la variable
# delai_traitement_jours
# ==========================================================

# Calcul du délai de traitement en jours
df["delai_traitement_jours"] = (
    df["date_traitement"] - df["date_demande"]
).dt.days

# Vérification des premières lignes
display(
    df[
        [
            "date_demande",
            "date_traitement",
            "delai_traitement_jours"
        ]
    ].head()
)

# Vérification du type et des statistiques
print("Type de la colonne :", df["delai_traitement_jours"].dtype)
print("\nStatistiques descriptives :")
display(df["delai_traitement_jours"].describe())

,date_demande,date_traitement,delai_traitement_jours
0,2026-01-14,2026-02-08,25
1,2026-05-31,2026-06-23,23
2,2026-06-16,2026-07-03,17
3,2026-06-07,2026-07-01,24
4,2026-02-04,2026-03-01,25


Type de la colonne : int64

Statistiques descriptives :


count    5000.000000
mean       19.453400
std         9.148342
min         1.000000
25%        12.000000
50%        20.000000
75%        27.000000
max        38.000000
Name: delai_traitement_jours, dtype: float64

## Enrichissement 2 : Création de la variable `delai_paiement_jours`

### Objectif

Cette étape consiste à créer une nouvelle variable dérivée nommée **`delai_paiement_jours`**, représentant le nombre de jours écoulés entre la date de traitement de la demande et sa date de paiement.

### Formule

$$
\text{delai\_paiement\_jours}
=
\text{date\_paiement}
-
\text{date\_traitement}
$$

### Justification métier

Cette variable permet d'évaluer la rapidité du processus de paiement après le traitement d'une demande.

Elle sera utilisée pour :

- mesurer les délais de paiement ;
- analyser l'efficacité du processus administratif et financier ;
- construire des indicateurs de performance (KPI) dans Power BI ;
- fournir une variable explicative pouvant être utilisée dans de futurs modèles de Machine Learning.

**Remarque :**

Les demandes ne possédant pas de date de paiement conservent une valeur manquante (`NaN`), ce qui indique qu'aucun délai de paiement ne peut être calculé.

In [36]:
# ==========================================================
# Enrichissement 2 : Création de la variable
# delai_paiement_jours
# ==========================================================

# Calcul du délai entre le traitement et le paiement
df["delai_paiement_jours"] = (
    df["date_paiement"] - df["date_traitement"]
).dt.days

# Vérification des premières lignes
display(
    df[
        [
            "date_traitement",
            "date_paiement",
            "delai_paiement_jours"
        ]
    ].head(10)
)

# Vérification du type et des statistiques
print("Type de la colonne :", df["delai_paiement_jours"].dtype)

print("\nStatistiques descriptives :")
display(df["delai_paiement_jours"].describe())

print("\nNombre de valeurs manquantes :", df["delai_paiement_jours"].isna().sum())

,date_traitement,date_paiement,delai_paiement_jours
0,2026-02-08,2026-02-09,1.0
1,2026-06-23,2026-06-23,0.0
2,2026-07-03,2026-07-04,1.0
3,2026-07-01,2026-07-05,4.0
4,2026-03-01,NaT,NaN
5,2026-06-24,2026-06-24,0.0
6,2026-01-13,NaT,NaN
7,2026-07-01,NaT,NaN
8,2026-01-13,2026-01-17,4.0
9,2026-07-01,2026-07-01,0.0


Type de la colonne : float64

Statistiques descriptives :


count    3171.000000
mean        2.868811
std         2.666738
min         0.000000
25%         1.000000
50%         2.000000
75%         4.000000
max        13.000000
Name: delai_paiement_jours, dtype: float64


Nombre de valeurs manquantes : 1829


## Enrichissement 3 : Création de la variable `anticipation_reservation_jours`

### Objectif

Cette étape consiste à créer une nouvelle variable dérivée nommée **`anticipation_reservation_jours`**, représentant le nombre de jours séparant la date de demande de la date de début du séjour.

### Formule

$$
\text{anticipation\_reservation\_jours}
=
\text{date\_debut\_sejour}
-
\text{date\_demande}
$$

### Justification métier

Cette variable mesure le délai d'anticipation avec lequel un agent effectue sa réservation.

Elle permettra de :

- analyser les habitudes de réservation des bénéficiaires ;
- identifier les périodes où les réservations sont effectuées très tôt ou très tard ;
- construire des indicateurs dans Power BI ;
- fournir une variable explicative pour les modèles de Machine Learning.

Le résultat est exprimé en **nombre de jours**.

In [37]:
# ==========================================================
# Enrichissement 3 : Création de la variable
# anticipation_reservation_jours
# ==========================================================

# Calcul du délai entre la demande et le début du séjour
df["anticipation_reservation_jours"] = (
    df["date_debut_sejour"] - df["date_demande"]
).dt.days

# Vérification
display(
    df[
        [
            "date_demande",
            "date_debut_sejour",
            "anticipation_reservation_jours"
        ]
    ].head()
)

print("Type :", df["anticipation_reservation_jours"].dtype)

display(df["anticipation_reservation_jours"].describe())

,date_demande,date_debut_sejour,anticipation_reservation_jours
0,2026-01-14,2026-02-21,38
1,2026-05-31,2026-07-15,45
2,2026-06-16,2026-07-31,45
3,2026-06-07,2026-07-31,54
4,2026-02-04,2026-03-09,33


Type : int64


count    5000.000000
mean       40.883000
std        18.262432
min         8.000000
25%        27.000000
50%        38.000000
75%        51.000000
max       102.000000
Name: anticipation_reservation_jours, dtype: float64

## Enrichissement 4 : Création de la variable `taux_prise_charge_ocp`

### Objectif

Cette étape consiste à créer une nouvelle variable dérivée nommée **`taux_prise_charge_ocp`**, représentant le pourcentage du coût total du séjour financé par OCP.

### Formule

$$
\text{taux\_prise\_charge\_ocp}
=
\frac{\text{montant\_pris\_charge\_ocp}}
{\text{montant\_total\_sejour}}
\times100
$$

### Justification métier

Cette variable permet de mesurer la part du coût total du séjour prise en charge par OCP.

Elle sera utilisée pour :

- analyser les dépenses supportées par OCP ;
- comparer les établissements selon leur niveau de prise en charge ;
- construire des indicateurs financiers dans Power BI ;
- enrichir les données pour de futures analyses prédictives.

Le résultat est exprimé en **pourcentage (%)**.

In [38]:
# ==========================================================
# Enrichissement 4 : Création de la variable
# taux_prise_charge_ocp
# ==========================================================

# Calcul du pourcentage de prise en charge par OCP
df["taux_prise_charge_ocp"] = (
    df["montant_pris_charge_ocp"] /
    df["montant_total_sejour"]
) * 100

# Arrondi à deux décimales
df["taux_prise_charge_ocp"] = df["taux_prise_charge_ocp"].round(2)

# Vérification
display(
    df[
        [
            "montant_total_sejour",
            "montant_pris_charge_ocp",
            "taux_prise_charge_ocp"
        ]
    ].head(10)
)

print("Type :", df["taux_prise_charge_ocp"].dtype)

display(df["taux_prise_charge_ocp"].describe())

,montant_total_sejour,montant_pris_charge_ocp,taux_prise_charge_ocp
0,12411.62,8837.40,71.20
1,4371.06,3055.88,69.91
2,2788.32,2271.02,81.45
3,13143.00,10167.39,77.36
4,8846.79,7286.41,82.36
5,22583.05,16859.01,74.65
6,2721.00,1769.91,65.05
7,17640.51,14391.79,81.58
8,12522.40,8814.58,70.39
9,28558.95,21015.97,73.59


Type : float64


count    5000.000000
mean       74.926872
std         5.771753
min        65.000000
25%        69.977500
50%        74.700000
75%        79.922500
max        84.990000
Name: taux_prise_charge_ocp, dtype: float64

## Enrichissement 5 : Création de la variable `taux_participation_agent`

### Objectif

Cette étape consiste à créer une nouvelle variable dérivée nommée **`taux_participation_agent`**, représentant le pourcentage du coût total du séjour financé par l'agent.

### Formule

$$
\text{taux\_participation\_agent}
=
\frac{\text{quote\_part\_agent}}
{\text{montant\_total\_sejour}}
\times100
$$

### Justification métier

Cette variable permet de mesurer la contribution financière de l'agent par rapport au coût total du séjour.

Elle sera utilisée pour :

- analyser la participation financière des bénéficiaires ;
- comparer la répartition des coûts entre l'agent et OCP ;
- construire des indicateurs financiers dans Power BI ;
- enrichir les données pour de futures analyses statistiques ou prédictives.

Le résultat est exprimé en **pourcentage (%)**.

In [39]:
# ==========================================================
# Enrichissement 5 : Création de la variable
# taux_participation_agent
# ==========================================================

# Calcul du pourcentage de participation de l'agent
df["taux_participation_agent"] = (
    df["quote_part_agent"] /
    df["montant_total_sejour"]
) * 100

# Arrondi à deux décimales
df["taux_participation_agent"] = df["taux_participation_agent"].round(2)

# Vérification
display(
    df[
        [
            "montant_total_sejour",
            "quote_part_agent",
            "taux_participation_agent"
        ]
    ].head()
)

print("Type :", df["taux_participation_agent"].dtype)

display(df["taux_participation_agent"].describe())

,montant_total_sejour,quote_part_agent,taux_participation_agent
0,12411.62,3574.22,28.80
1,4371.06,1315.18,30.09
2,2788.32,517.30,18.55
3,13143.00,2975.61,22.64
4,8846.79,1560.38,17.64


Type : float64


count    5000.000000
mean       25.073128
std         5.771753
min        15.010000
25%        20.077500
50%        25.300000
75%        30.022500
max        35.000000
Name: taux_participation_agent, dtype: float64

## Enrichissement 6 : Création de la variable `cout_par_personne`

### Objectif

Cette étape consiste à créer une nouvelle variable dérivée nommée **`cout_par_personne`**, représentant le coût moyen du séjour par membre de la famille.

### Formule

$$
\text{cout\_par\_personne}
=
\frac{\text{montant\_total\_sejour}}
{\text{total\_membres\_famille}}
$$

### Justification métier

Le coût total d'un séjour dépend du nombre de personnes concernées.

Cette variable permet de comparer les séjours sur une base équitable en calculant le coût moyen par bénéficiaire.

Elle sera utilisée pour :

- analyser les coûts moyens par personne ;
- comparer les établissements ;
- construire des indicateurs financiers dans Power BI ;
- servir de variable explicative dans les modèles de Machine Learning.

Le résultat est exprimé en **dirhams par personne**.

In [40]:
# ==========================================================
# Enrichissement 6 : Création de la variable
# cout_par_personne
# ==========================================================

# Calcul du coût moyen par personne
df["cout_par_personne"] = (
    df["montant_total_sejour"] /
    df["total_membres_famille"]
)

# Arrondi à deux décimales
df["cout_par_personne"] = df["cout_par_personne"].round(2)

# Vérification
display(
    df[
        [
            "montant_total_sejour",
            "total_membres_famille",
            "cout_par_personne"
        ]
    ].head()
)

print("Type :", df["cout_par_personne"].dtype)

display(df["cout_par_personne"].describe())

,montant_total_sejour,total_membres_famille,cout_par_personne
0,12411.62,3,4137.21
1,4371.06,2,2185.53
2,2788.32,1,2788.32
3,13143.00,3,4381.00
4,8846.79,3,2948.93


Type : float64


count    5000.000000
mean     2759.255936
std       915.330872
min      1026.560000
25%      2080.375000
50%      2618.685000
75%      3271.642500
max      7170.070000
Name: cout_par_personne, dtype: float64

## Enrichissement 7 : Création de la variable `categorie_taille_groupe`

### Objectif

Cette étape consiste à créer une nouvelle variable catégorielle nommée **`categorie_taille_groupe`**, permettant de regrouper les bénéficiaires selon la taille de leur groupe familial.

### Règles de catégorisation

- **1 personne** : Individuel
- **2 personnes** : Couple
- **3 à 4 personnes** : Petite famille
- **5 personnes et plus** : Grande famille

### Justification métier

Cette variable facilite la segmentation des bénéficiaires et simplifie les analyses dans Power BI.

Elle permettra notamment de :

- analyser la répartition des types de groupes ;
- comparer les coûts selon la taille des familles ;
- étudier les besoins en capacité d'hébergement ;
- enrichir les variables utilisées dans les modèles de Machine Learning.

In [41]:
# ==========================================================
# Enrichissement 7 : Création de la variable
# categorie_taille_groupe
# ==========================================================

def categoriser_taille_groupe(nb):
    if nb == 1:
        return "Individuel"
    elif nb == 2:
        return "Couple"
    elif 3 <= nb <= 4:
        return "Petite famille"
    else:
        return "Grande famille"


df["categorie_taille_groupe"] = (
    df["total_membres_famille"]
    .apply(categoriser_taille_groupe)
)

# Vérification
display(
    df[
        [
            "total_membres_famille",
            "categorie_taille_groupe"
        ]
    ].head(10)
)

print("Répartition des catégories :")
display(df["categorie_taille_groupe"].value_counts())

,total_membres_famille,categorie_taille_groupe
0,3,Petite famille
1,2,Couple
2,1,Individuel
3,3,Petite famille
4,3,Petite famille
5,4,Petite famille
6,1,Individuel
7,7,Grande famille
8,5,Grande famille
9,5,Grande famille


Répartition des catégories :


categorie_taille_groupe
Petite famille    1831
Grande famille    1307
Individuel        1203
Couple             659
Name: count, dtype: int64

## Enrichissement 9 : Création de la variable `categorie_anciennete`

### Objectif

Cette étape consiste à créer une nouvelle variable catégorielle nommée **`categorie_anciennete`**, permettant de regrouper les agents selon leur ancienneté.

### Règles de catégorisation

- **0 à 5 ans** : Débutant
- **6 à 15 ans** : Intermédiaire
- **16 ans et plus** : Expérimenté

### Justification métier

Cette variable facilite la segmentation des agents selon leur expérience professionnelle.

Elle permettra de :

- comparer les comportements de réservation selon l'ancienneté ;
- analyser les dépenses en fonction du profil des agents ;
- créer des visualisations plus lisibles dans Power BI ;
- enrichir les variables utilisées dans les modèles de Machine Learning.

In [42]:
# ==========================================================
# Enrichissement 9 : Création de la variable
# categorie_anciennete
# ==========================================================

def categoriser_anciennete(anciennete):
    if anciennete <= 5:
        return "Débutant"
    elif anciennete <= 15:
        return "Intermédiaire"
    else:
        return "Expérimenté"


df["categorie_anciennete"] = (
    df["anciennete_annees"]
    .apply(categoriser_anciennete)
)

# Vérification
display(
    df[
        [
            "anciennete_annees",
            "categorie_anciennete"
        ]
    ].head(10)
)

print("Répartition des catégories :")
display(df["categorie_anciennete"].value_counts())

,anciennete_annees,categorie_anciennete
0,2,Débutant
1,12,Intermédiaire
2,1,Débutant
3,8,Intermédiaire
4,5,Débutant
5,35,Expérimenté
6,2,Débutant
7,1,Débutant
8,5,Débutant
9,1,Débutant


Répartition des catégories :


categorie_anciennete
Débutant         2465
Intermédiaire    1697
Expérimenté       838
Name: count, dtype: int64

# Sauvegarde du jeu de données enrichi

À l'issue des différentes étapes d'enrichissement, le jeu de données est enregistré dans un nouveau fichier CSV.

Ce fichier constitue la version finale enrichie qui sera utilisée dans les prochaines étapes du projet :

- importation dans SQL Server ;
- création du schéma en étoile ;
- développement des tableaux de bord Power BI ;
- mise en œuvre des modèles de Machine Learning.

In [44]:
# ==========================================================
# Sauvegarde du dataset enrichi
# ==========================================================

nom_fichier = "dataset_ocp_social_2026_nettoye_enrichi_final_version2.csv"

df.to_csv(
    nom_fichier,
    index=False,
    encoding="utf-8-sig"
)

print("===========================================")
print("Enrichissement terminé avec succès")
print("===========================================")
print(f"Fichier enregistré : {nom_fichier}")
print(f"Nombre de lignes    : {df.shape[0]}")
print(f"Nombre de colonnes  : {df.shape[1]}")

print("\nNouvelles variables créées :")

nouvelles_colonnes = [
    "delai_traitement_jours",
    "delai_paiement_jours",
    "anticipation_reservation_jours",
    "taux_prise_charge_ocp",
    "taux_participation_agent",
    "cout_par_personne",
    "categorie_taille_groupe",
    "categorie_anciennete"
]

for colonne in nouvelles_colonnes:
    print(f"• {colonne}")

Enrichissement terminé avec succès
Fichier enregistré : dataset_ocp_social_2026_nettoye_enrichi_final_version2.csv
Nombre de lignes    : 5000
Nombre de colonnes  : 55

Nouvelles variables créées :
• delai_traitement_jours
• delai_paiement_jours
• anticipation_reservation_jours
• taux_prise_charge_ocp
• taux_participation_agent
• cout_par_personne
• categorie_taille_groupe
• categorie_anciennete
